# 02 — Sequential Pattern Mining with PrefixSpan

This notebook performs the sequential pattern mining stage of the sepsis project.

It loads the symbolic patient sequences generated by Notebook 01 and mines frequent temporal patterns separately for:
- the pre-sepsis positive cohort
- the non-septic negative cohort

The primary algorithm is PrefixSpan.

This notebook does **not** perform closed-pattern filtering, discriminative pattern selection, sequence-distance calculation, classification, or final evaluation. Those stages are handled later.

## 1. Imports and Configuration

In [ ]:
from pathlib import Path
import pickle
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve().parent
SEQUENCE_DIR = PROJECT_ROOT / "outputs" / "sequences"
PATTERN_DIR = PROJECT_ROOT / "outputs" / "patterns"
PATTERN_DIR.mkdir(parents=True, exist_ok=True)

POSITIVE_PATH = SEQUENCE_DIR / "positive_sequences.pkl"
NEGATIVE_PATH = SEQUENCE_DIR / "negative_sequences.pkl"

# Main PrefixSpan parameters.
MIN_SUPPORT = 0.05
MIN_PATTERN_LENGTH = 2
MAX_PATTERN_LENGTH = 5

# Development controls. Set to None for the complete selected training set.
MAX_POSITIVE_SEQUENCES = None
MAX_NEGATIVE_SEQUENCES = None

print("Sequence directory:", SEQUENCE_DIR)
print("Pattern output directory:", PATTERN_DIR)
print("Minimum support:", MIN_SUPPORT)
print("Pattern length:", MIN_PATTERN_LENGTH, "to", MAX_PATTERN_LENGTH)


## 2. Load Symbolic Sequences

In [ ]:
if not POSITIVE_PATH.exists():
    raise FileNotFoundError(
        f"Missing {POSITIVE_PATH}. Run 01_Preprocessing_and_Sequence_Building.ipynb first."
    )
if not NEGATIVE_PATH.exists():
    raise FileNotFoundError(
        f"Missing {NEGATIVE_PATH}. Run 01_Preprocessing_and_Sequence_Building.ipynb first."
    )

with open(POSITIVE_PATH, "rb") as f:
    positive_records = pickle.load(f)

with open(NEGATIVE_PATH, "rb") as f:
    negative_records = pickle.load(f)

print(f"Positive patient sequences: {len(positive_records):,}")
print(f"Negative patient sequences: {len(negative_records):,}")


## 3. Convert Patient Records to PrefixSpan Input

Each patient is represented as an ordered sequence of hourly itemsets:

```text
[
    [HR_HIGH, MAP_NORMAL, TEMP_HIGH],
    [HR_HIGH, MAP_LOW],
    [RESP_HIGH, MAP_LOW, TEMP_HIGH],
    ...
]
```

The order of the hourly itemsets is temporal. Symbols inside one hourly itemset are treated as co-occurring items rather than as a temporal sequence.

In [ ]:
def extract_sequences(records, max_sequences=None):
    if max_sequences is not None:
        records = records[:max_sequences]
    return [record["sequence"] for record in records]

positive_sequences = extract_sequences(positive_records, MAX_POSITIVE_SEQUENCES)
negative_sequences = extract_sequences(negative_records, MAX_NEGATIVE_SEQUENCES)

print("Positive sequences:", len(positive_sequences))
print("Negative sequences:", len(negative_sequences))

if positive_sequences:
    print("\nExample positive sequence:")
    for hour, itemset in enumerate(positive_sequences[0], start=1):
        print(f"Hour {hour}: {itemset}")


## 4. Sequence Statistics

In [ ]:
def sequence_statistics(sequences):
    lengths = np.array([len(seq) for seq in sequences], dtype=int)
    item_counts = []
    unique_items = set()

    for seq in sequences:
        count = 0
        for itemset in seq:
            count += len(itemset)
            unique_items.update(itemset)
        item_counts.append(count)

    return {
        "sequences": len(sequences),
        "mean_sequence_length": lengths.mean() if len(lengths) else 0,
        "median_sequence_length": np.median(lengths) if len(lengths) else 0,
        "max_sequence_length": lengths.max() if len(lengths) else 0,
        "mean_items_per_sequence": np.mean(item_counts) if item_counts else 0,
        "unique_symbols": len(unique_items),
    }

stats = pd.DataFrame([
    {"Cohort": "Positive", **sequence_statistics(positive_sequences)},
    {"Cohort": "Negative", **sequence_statistics(negative_sequences)},
])

display(stats)


## 5. Import PrefixSpan

This notebook uses the lightweight `prefixspan` Python package.

If it is not installed, install it once in the project environment:

```bash
pip install prefixspan
```

Do not repeatedly install packages from inside the notebook.

In [ ]:
try:
    from prefixspan import PrefixSpan
    print("prefixspan imported successfully.")
except ImportError as exc:
    raise ImportError(
        "The 'prefixspan' package is not installed in the active environment. "
        "Install it once with: pip install prefixspan"
    ) from exc


## 6. Prepare the PrefixSpan Representation

In [ ]:
def prepare_for_prefixspan(sequences):
    prepared = []

    for sequence in sequences:
        itemsets = []
        for itemset in sequence:
            cleaned = tuple(sorted(set(itemset)))
            if cleaned:
                itemsets.append(cleaned)
        if itemsets:
            prepared.append(itemsets)

    return prepared

positive_ps = prepare_for_prefixspan(positive_sequences)
negative_ps = prepare_for_prefixspan(negative_sequences)

print("Prepared positive sequences:", len(positive_ps))
print("Prepared negative sequences:", len(negative_ps))


## 7. Convert Minimum Support to a Count

In [ ]:
def support_to_count(min_support, n_sequences):
    if not 0 < min_support <= 1:
        raise ValueError("MIN_SUPPORT must be between 0 and 1.")
    return max(1, int(np.ceil(min_support * n_sequences)))

positive_min_count = support_to_count(MIN_SUPPORT, len(positive_ps))
negative_min_count = support_to_count(MIN_SUPPORT, len(negative_ps))

print(f"Positive minimum support count: {positive_min_count:,}")
print(f"Negative minimum support count: {negative_min_count:,}")


## 8. Run PrefixSpan on the Positive / Pre-Sepsis Cohort

In [ ]:
def run_prefixspan(sequences, min_support_count):
    start = time.perf_counter()
    miner = PrefixSpan(sequences)
    results = miner.frequent(min_support_count)
    elapsed = time.perf_counter() - start
    return results, elapsed

positive_raw_patterns, positive_time = run_prefixspan(
    positive_ps, positive_min_count
)

print(f"Positive PrefixSpan runtime: {positive_time:.2f} seconds")
print(f"Positive frequent patterns: {len(positive_raw_patterns):,}")


## 9. Run PrefixSpan on the Negative Cohort

In [ ]:
negative_raw_patterns, negative_time = run_prefixspan(
    negative_ps, negative_min_count
)

print(f"Negative PrefixSpan runtime: {negative_time:.2f} seconds")
print(f"Negative frequent patterns: {len(negative_raw_patterns):,}")


## 10. Convert PrefixSpan Results to Standard Tables

In [ ]:
def pattern_to_text(pattern):
    return " → ".join(
        " + ".join(itemset) if isinstance(itemset, (tuple, list)) else str(itemset)
        for itemset in pattern
    )

def prefixspan_to_dataframe(results, n_sequences, cohort):
    rows = []

    for support_count, pattern in results:
        if not (MIN_PATTERN_LENGTH <= len(pattern) <= MAX_PATTERN_LENGTH):
            continue

        rows.append({
            "cohort": cohort,
            "pattern": tuple(pattern),
            "pattern_text": pattern_to_text(pattern),
            "pattern_length": len(pattern),
            "support_count": int(support_count),
            "support": float(support_count / n_sequences),
        })

    return pd.DataFrame(rows)

positive_patterns = prefixspan_to_dataframe(
    positive_raw_patterns, len(positive_ps), "positive"
)
negative_patterns = prefixspan_to_dataframe(
    negative_raw_patterns, len(negative_ps), "negative"
)

print("Filtered positive patterns:", len(positive_patterns))
print("Filtered negative patterns:", len(negative_patterns))

display(positive_patterns.head(20))


## 11. Most Frequent Positive / Pre-Sepsis Patterns

In [ ]:
if positive_patterns.empty:
    print("No positive patterns passed the current support and length settings.")
else:
    top_positive = (
        positive_patterns
        .sort_values(["support", "pattern_length"], ascending=[False, True])
        .head(25)
    )
    display(top_positive[
        ["pattern_text", "pattern_length", "support_count", "support"]
    ])


## 12. Most Frequent Negative Patterns

In [ ]:
if negative_patterns.empty:
    print("No negative patterns passed the current support and length settings.")
else:
    top_negative = (
        negative_patterns
        .sort_values(["support", "pattern_length"], ascending=[False, True])
        .head(25)
    )
    display(top_negative[
        ["pattern_text", "pattern_length", "support_count", "support"]
    ])


## 13. Pattern-Length Distribution

In [ ]:
combined_patterns = pd.concat(
    [positive_patterns, negative_patterns],
    ignore_index=True
)

if not combined_patterns.empty:
    counts = (
        combined_patterns
        .groupby(["cohort", "pattern_length"])
        .size()
        .reset_index(name="count")
    )

    plt.figure(figsize=(9, 5))

    for cohort in ["positive", "negative"]:
        subset = counts[counts["cohort"] == cohort]
        plt.plot(
            subset["pattern_length"],
            subset["count"],
            marker="o",
            label=cohort.capitalize()
        )

    plt.title("Number of Frequent Patterns by Pattern Length")
    plt.xlabel("Sequential pattern length")
    plt.ylabel("Number of patterns")
    plt.xticks(range(MIN_PATTERN_LENGTH, MAX_PATTERN_LENGTH + 1))
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        PATTERN_DIR / "01_pattern_length_distribution.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()


## 14. Pattern Support Distribution

In [ ]:
if not combined_patterns.empty:
    plt.figure(figsize=(9, 5))

    for cohort in ["positive", "negative"]:
        subset = combined_patterns[combined_patterns["cohort"] == cohort]
        if not subset.empty:
            plt.hist(
                subset["support"],
                bins=30,
                alpha=0.6,
                label=cohort.capitalize()
            )

    plt.title("Support Distribution of Frequent Sequential Patterns")
    plt.xlabel("Support")
    plt.ylabel("Number of patterns")
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        PATTERN_DIR / "02_pattern_support_distribution.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()


## 15. Top Positive Patterns by Support

In [ ]:
if not positive_patterns.empty:
    top = (
        positive_patterns
        .sort_values("support", ascending=False)
        .head(15)
        .sort_values("support")
    )

    plt.figure(figsize=(11, 7))
    plt.barh(top["pattern_text"], top["support"])
    plt.title("Top Frequent Pre-Sepsis Sequential Patterns")
    plt.xlabel("Support")
    plt.ylabel("Pattern")
    plt.tight_layout()
    plt.savefig(
        PATTERN_DIR / "03_top_positive_patterns.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()


## 16. Top Negative Patterns by Support

In [ ]:
if not negative_patterns.empty:
    top = (
        negative_patterns
        .sort_values("support", ascending=False)
        .head(15)
        .sort_values("support")
    )

    plt.figure(figsize=(11, 7))
    plt.barh(top["pattern_text"], top["support"])
    plt.title("Top Frequent Negative-Cohort Sequential Patterns")
    plt.xlabel("Support")
    plt.ylabel("Pattern")
    plt.tight_layout()
    plt.savefig(
        PATTERN_DIR / "04_top_negative_patterns.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()


## 17. Positive-vs-Negative Support Comparison

This is only a preliminary comparison. Notebook 03 performs the actual closed-pattern and discriminative-pattern selection.

In [ ]:
positive_lookup = positive_patterns.set_index("pattern")["support"].to_dict()
negative_lookup = negative_patterns.set_index("pattern")["support"].to_dict()

all_patterns = set(positive_lookup) | set(negative_lookup)

comparison_rows = []

for pattern in all_patterns:
    pos_support = positive_lookup.get(pattern, 0.0)
    neg_support = negative_lookup.get(pattern, 0.0)

    comparison_rows.append({
        "pattern": pattern,
        "pattern_text": pattern_to_text(pattern),
        "positive_support": pos_support,
        "negative_support": neg_support,
        "support_difference": pos_support - neg_support,
    })

pattern_comparison = pd.DataFrame(comparison_rows)

if not pattern_comparison.empty:
    pattern_comparison = pattern_comparison.sort_values(
        "support_difference",
        ascending=False
    )

display(pattern_comparison.head(25))


## 18. Preliminary Support-Difference Plot

In [ ]:
if not pattern_comparison.empty:
    top = pattern_comparison.head(15).sort_values("support_difference")

    plt.figure(figsize=(11, 7))
    plt.barh(top["pattern_text"], top["support_difference"])
    plt.axvline(0, linestyle="--")
    plt.title("Support Difference for Frequent Sequential Patterns")
    plt.xlabel("Positive support − negative support")
    plt.ylabel("Pattern")
    plt.tight_layout()
    plt.savefig(
        PATTERN_DIR / "05_support_difference.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()


## 19. Save Frequent Pattern Results

In [ ]:
positive_output = PATTERN_DIR / "frequent_positive_patterns.csv"
negative_output = PATTERN_DIR / "frequent_negative_patterns.csv"
comparison_output = PATTERN_DIR / "pattern_support_comparison.csv"

positive_patterns.to_csv(positive_output, index=False)
negative_patterns.to_csv(negative_output, index=False)
pattern_comparison.to_csv(comparison_output, index=False)

run_summary = pd.DataFrame({
    "cohort": ["positive", "negative"],
    "input_sequences": [len(positive_ps), len(negative_ps)],
    "min_support": [MIN_SUPPORT, MIN_SUPPORT],
    "min_support_count": [positive_min_count, negative_min_count],
    "raw_patterns": [
        len(positive_raw_patterns),
        len(negative_raw_patterns)
    ],
    "filtered_patterns": [
        len(positive_patterns),
        len(negative_patterns)
    ],
    "runtime_seconds": [positive_time, negative_time],
})

run_summary.to_csv(PATTERN_DIR / "prefixspan_run_summary.csv", index=False)

display(run_summary)

print("Saved:")
print(" -", positive_output)
print(" -", negative_output)
print(" -", comparison_output)
print(" -", PATTERN_DIR / "prefixspan_run_summary.csv")


## 20. Sanity Checks

Before proceeding to Notebook 03, verify that:
- both cohorts were mined
- support values are valid
- retained patterns satisfy the requested length range
- output files were created

Do not interpret the highest-support patterns as clinically important yet. The next notebook will perform closed-pattern filtering and discriminative comparison.

In [ ]:
assert len(positive_ps) > 0, "No positive sequences available."
assert len(negative_ps) > 0, "No negative sequences available."

for df in [positive_patterns, negative_patterns]:
    if not df.empty:
        assert df["support"].between(0, 1).all()
        assert df["pattern_length"].between(
            MIN_PATTERN_LENGTH,
            MAX_PATTERN_LENGTH
        ).all()
        assert (df["support_count"] >= 1).all()

assert positive_output.exists()
assert negative_output.exists()

print("PrefixSpan notebook sanity checks passed.")
